<a href="https://colab.research.google.com/github/sanmeet1811/Python-Projects/blob/main/Commodity_Forcast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Runs the full commodity analysis pipeline:
  Step 1: Fetch price data
  Step 2: Compute indicators
  Step 3: Forecast prices
  Step 4: Generate charts
  Step 5: Print analyst summary report

In [9]:
import numpy as np

Loading data from Yahoo Finance directly

In [19]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import os

COMMODITIES = {
    'Crude_Oil':   'CL=F',
    'Gold':        'GC=F',
    'Natural_Gas': 'NG=F',
    'Silver':      'SI=F',
    'Wheat':       'ZW=F',
}

save_folder = 'data'
os.makedirs(save_folder, exist_ok=True)

end_date   = datetime.today()
start_date = end_date - timedelta(days=365 * 2)

price_data = {}

for name, ticker in COMMODITIES.items():
    try:
        raw = yf.download(ticker, start=start_date, end=end_date,
                          progress=False, auto_adjust=True)
        if raw.empty:
            print(f"✗ {name} — no data returned")
            continue

        close = raw['Close']

        # Fix: flatten if yfinance returns a DataFrame instead of Series
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]

        close.name = name
        price_data[name] = close
        print(f"✓ {name}  ({len(close)} rows)")

    except Exception as e:
        print(f"✗ {name} — error: {e}")

# Combine safely
df = pd.concat(price_data.values(), axis=1)
df.columns = price_data.keys()
df = df.ffill().dropna()

# Save
df.to_csv(f"{save_folder}/all_commodities.csv")
print(f"\nShape : {df.shape}")
df.tail(3)

✓ Crude_Oil  (502 rows)
✓ Gold  (502 rows)
✓ Natural_Gas  (502 rows)
✓ Silver  (502 rows)
✓ Wheat  (501 rows)

Shape : (502, 5)


,Crude_Oil,Gold,Natural_Gas,Silver,Wheat
Date,,,,,
2026-04-23,95.849998,4705.100098,2.614,75.464996,610.75
2026-04-24,94.400002,4722.299805,2.523,76.383003,608.25
2026-04-27,96.830002,4728.500000,2.698,75.730003,625.00


In [6]:
!pip install yfinance pandas numpy matplotlib seaborn scikit-learn

#Analysis

In [30]:
def compute_moving_averages(df, short=20, long=50):
    """
    Adds 20-day and 50-day moving averages for every commodity.
    Returns a dictionary: { 'Crude_Oil': DataFrame_with_MAs, ... }
    """
    results = {}

    for col in df.columns:
        series = df[col].dropna()
        temp = pd.DataFrame({'Price': series})
        temp[f'MA_{short}']  = series.rolling(window=short).mean()
        temp[f'MA_{long}']   = series.rolling(window=long).mean()
        temp['Signal'] = np.where(temp[f'MA_{short}'] > temp[f'MA_{long}'], 'Bullish', 'Bearish')
        results[col] = temp

    return results

In [31]:
def compute_daily_returns(df):
    """
    Computes percentage daily returns.
    Returns a DataFrame of returns.
    """
    returns = df.pct_change().dropna()
    returns.columns = [f"{c}_Return" for c in returns.columns]
    return returns


In [32]:
def compute_volatility(df, window=30):
    """
    Computes rolling annualised volatility (std of daily returns).
    Uses 252 trading days convention.
    """
    daily_returns = df.pct_change()
    volatility    = daily_returns.rolling(window=window).std() * np.sqrt(252) * 100
    volatility.columns = [f"{c}_Volatility_%" for c in volatility.columns]
    return volatility.dropna()

In [33]:
def compute_correlation(df):
    """
    Returns the correlation matrix between all commodities.
    """
    daily_returns = df.pct_change().dropna()
    corr_matrix   = daily_returns.corr().round(3)
    return corr_matrix


In [35]:
def print_summary(df):
    """
    Prints a readable summary table for the latest available prices.
    """
    latest = df.iloc[-1]
    prev   = df.iloc[-2]
    returns_today = ((latest - prev) / prev * 100).round(2)

    print(f"  {'Commodity':<18} {'Price':>10}  {'1D Change':>10}")
    for col in df.columns:
        change = returns_today[col]
        arrow  = "up" if change >= 0 else "down"
        print(f"  {col:<18} {latest[col]:>10.2f}  {arrow} {abs(change):>6.2f}%")


In [36]:
print_summary(df)

  Commodity               Price   1D Change
  Crude_Oil               96.83  up   2.57%
  Gold                  4728.50  up   0.13%
  Natural_Gas              2.70  up   6.94%
  Silver                  75.73  down   0.85%
  Wheat                  625.00  up   2.75%


#Forecast
Generates a 30-day price forecast for each commodity
using linear regression on the most recent 60 days of prices.

In [43]:
from sklearn.linear_model import LinearRegression
from datetime import timedelta

In [50]:
from sklearn.linear_model import LinearRegression
import numpy as np
from datetime import timedelta

def forecast_price(series, lookback=60, horizon=30):
    series = series.dropna()

    if len(series) < lookback:
        print(f"  [!] Not enough data for forecast (need {lookback}, got {len(series)})")
        return None

    # ← everything below must be indented INSIDE the function
    recent = series.iloc[-lookback:]          # note: -lookback not lookback
    X = np.arange(len(recent)).reshape(-1, 1)
    y = recent.values

    model = LinearRegression()
    model.fit(X, y)

    y_hat        = model.predict(X)
    residual_std = np.std(y - y_hat)

    future_X     = np.arange(len(recent), len(recent) + horizon).reshape(-1, 1)
    forecast     = model.predict(future_X)

    last_date    = series.index[-1]
    future_dates = [last_date + timedelta(days=i+1) for i in range(horizon)]

    result = pd.DataFrame({
        'Forecast': forecast,
        'Lower':    forecast - residual_std,
        'Upper':    forecast + residual_std,
    }, index=future_dates)

    trend = 'Bullish 📈' if model.coef_[0] > 0 else 'Bearish 📉'

    return result, model.coef_[0], trend

In [52]:
def forecast_all(df, lookback=60, horizon=30):
    """
    Runs forecasts for all commodities and returns a dict of results.
    """
    forecasts = {}


    print(f"  30-Day Price Forecasts (based on last {lookback} trading days)")

    print(f"  {'Commodity':<18} {'Current':>9}  {'30D Forecast':>13}  {'Trend':>12}")


    for col in df.columns:
        out = forecast_price(df[col], lookback=lookback, horizon=horizon)
        if out is None:
            continue

        result_df, slope, trend = out
        current  = df[col].dropna().iloc[-1]
        projected = result_df['Forecast'].iloc[-1]

        print(f"  {col:<18} {current:>9.2f}  {projected:>13.2f}  {trend:>12}")

        forecasts[col] = {
            'data':    result_df,
            'slope':   slope,
            'trend':   trend,
            'current': current,
            'target':  projected,
        }

    return forecasts

In [53]:
forecast_all(df)

  30-Day Price Forecasts (based on last 60 trading days)
  Commodity            Current   30D Forecast         Trend
  Crude_Oil              96.83         128.56     Bullish 📈
  Gold                 4728.50        4477.72     Bearish 📉
  Natural_Gas             2.70           2.22     Bearish 📉
  Silver                 75.73          71.00     Bearish 📉
  Wheat                 625.00         654.60     Bullish 📈


{'Crude_Oil': {'data':               Forecast       Lower       Upper
  2026-04-28  106.848983   98.249754  115.448213
  2026-04-29  107.597622   98.998392  116.196852
  2026-04-30  108.346261   99.747031  116.945491
  2026-05-01  109.094900  100.495670  117.694130
  2026-05-02  109.843539  101.244309  118.442768
  2026-05-03  110.592177  101.992948  119.191407
  2026-05-04  111.340816  102.741586  119.940046
  2026-05-05  112.089455  103.490225  120.688685
  2026-05-06  112.838094  104.238864  121.437324
  2026-05-07  113.586733  104.987503  122.185963
  2026-05-08  114.335371  105.736142  122.934601
  2026-05-09  115.084010  106.484780  123.683240
  2026-05-10  115.832649  107.233419  124.431879
  2026-05-11  116.581288  107.982058  125.180518
  2026-05-12  117.329927  108.730697  125.929157
  2026-05-13  118.078566  109.479336  126.677795
  2026-05-14  118.827204  110.227974  127.426434
  2026-05-15  119.575843  110.976613  128.175073
  2026-05-16  120.324482  111.725252  128.923712